<!--
Copyright Advanced Micro Devices, Inc., or its affiliates.
SPDX-License-Identifier: MIT
-->

# Subtiling as layouts — the affine half of a GEMM kernel

AMD's **tensilelite** (hipBLASLt) generates GEMM kernels by *subtiling*: a macro
tile is carved into MMA tiles, those are split across waves, each lane reads a
fragment, and global/LDS read offsets are computed per lane. A surprising amount
of that bookkeeping is **affine** — a coordinate maps to an offset by
`Σ coordᵢ · strideᵢ` — which is *exactly* what a mosaic `Layout` is.

This notebook puts each affine subtiling map **next to the tensilelite code it
comes from** (pinned to commit `0d228e9`), then shows the mosaic `Layout` that
reproduces it — using **just the compiled bindings** (`Layout`,
`blocked_product`, `idx2crd`, `pretty_print`): we evaluate the map numerically
and *draw* the tiling, no extra dependencies required.

The cases that are **not** affine — LDS bank-conflict swizzles, cooperative-load
splitting, the fully-fused lane∘subtile∘wave address — are cataloged at the end,
each with the tensilelite snippet and the mosaic primitive it would need.

> Sizes are scaled down (e.g. a 16×instK MMA tile drawn as 2×2) so the offset
> grids stay readable. The shape/stride *algebra* is identical to the real
> kernel; only the magnitudes change.

In [1]:
import pathlib
import sys


def _ensure_pymosaic():
    """Import pymosaic from an install or the in-repo source checkout.

    Uses ``blocked_product`` as a recency canary so a stale kernel that cached
    an older pymosaic self-heals on re-run.
    """
    def _fresh_enough(mod):
        return hasattr(mod, "Layout") and hasattr(mod, "blocked_product")

    try:
        import pymosaic
        if _fresh_enough(pymosaic):
            return pymosaic
    except ImportError:
        pass

    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        pkg = cand / "python" / "pymosaic"
        if (pkg / "pymosaic" / "__init__.py").exists():
            sys.path.insert(0, str(pkg))
            break
    for name in [m for m in list(sys.modules) if m == "pymosaic" or m.startswith("pymosaic.")]:
        del sys.modules[name]
    import pymosaic
    return pymosaic


pymosaic = _ensure_pymosaic()
from pymosaic import Layout, blocked_product, coalesce


def is_perm(L):
    """True iff the layout visits every offset in [0, size) exactly once."""
    mapped = [L[i] for i in range(len(L))]
    return sorted(mapped) == list(range(len(L)))


print("pymosaic", pymosaic.__version__, "ready")

pymosaic 0.1.0 ready


## The map: which subtiling jobs are affine?

| Bucket | Subtiling job | tensilelite source | Affine? | § |
|---|---|---|---|---|
| a | macro-tile → MMA-tile grid | `SubtileGeometry.globalMMATileGrid` | ✅ | 1 |
| a | GR subtile footprint (strided strips) | `SubtileGeometry.ABGRGeometry` | ✅ | 2 |
| c | wave-group partition | `SubtileGeometry.localMMATileGrid` | ✅ | 3 |
| g | C/D epilogue store tile | `SubtileGeometry.CDTileGeometry` | ✅ | 4 |
| d | MMA lane → element fragment | `SubtileGeometry.MMALayout` | ✅ | 5 |
| b | global-read thread → byte offset | `SubtileGREmit._emitGROffset_TLU0` | ✅ skeleton | 6 |
| e | local-read (LDS) byte offset | `SubtileLREmit._emitLROffset_TLU0` | ✅ skeleton | 7 |
| h | wgid → output coordinate | workgroup schedule decode | ✅ | 8 |
| e | LDS bank-conflict **swizzle** | `SubtileGREmit` steps 2–4 | ❌ XOR/permute | catalog |
| c | cooperative-load split (`loadRatioGR`) | `TileInfo.grLoadIndexForSubtile` | ❌ divide | catalog |
| f | MX scale `op_sel` selection | `Kernel.emitMfmaCode` | ❌ bit-interleave | catalog |
| b+c+d | fully-fused lane∘subtile∘wave map | the whole offset path | ❌ composition | catalog |

The eight ✅ rows follow; the ❌ rows close the notebook.

## 1. Macro-tile → MMA-tile grid  *(bucket a)*

```python
# tensilelite — SubtileGeometry.py: TileGeometry.globalMMATileGrid
def globalMMATileGrid(self, macroTile, depthU):
    return (macroTile // self.mmaTileShape[0],   # MT     / 16
            depthU    // self.mmaTileShape[1])    # depthU / instK
```

That partitions the macro tile into a grid of 16×instK MMA tiles — a **tiling**:
drop a copy of one MMA tile at every grid position, i.e.
`blocked_product(tile, grid)`.

In [2]:
# A real MMA tile is 16 (M) x instK (K, =32 for bf16). Scaled to 2x2 here so
# the grid is readable; the shape/stride algebra is identical.
mma_tile  = Layout.row_major(2, 2)        # one MMA tile  (rows = M, cols = K)
tile_grid = Layout.column_major(2, 3)     # globalMMATileGrid = (MT/16, depthU/instK)

macro = blocked_product(mma_tile, tile_grid)
print("macro layout:", macro, "| grid:", tile_grid.shape, "| permutation:", is_perm(macro))
macro.pretty_print()

macro layout: ((2,2),(2,3)):((2,4),(1,8)) | grid: (2, 3) | permutation: True
((2,2),(2,3)):((2,4),(1,8))   size=24  cosize=24
        (0,0)   (1,0)   (0,1)   (1,1)   (0,2)   (1,2)
      ┌───────┬───────╥───────┬───────╥───────┬───────┐
(0,0) │     0 │     1 ║     8 │     9 ║    16 │    17 │
      ├───────┼───────╫───────┼───────╫───────┼───────┤
(1,0) │     2 │     3 ║    10 │    11 ║    18 │    19 │
      ╞═══════╪═══════╬═══════╪═══════╬═══════╪═══════╡
(0,1) │     4 │     5 ║    12 │    13 ║    20 │    21 │
      ├───────┼───────╫───────┼───────╫───────┼───────┤
(1,1) │     6 │     7 ║    14 │    15 ║    22 │    23 │
      └───────┴───────╨───────┴───────╨───────┴───────┘


## 2. GR subtile footprint — strided strips  *(bucket a)*

`ABGRGeometry`'s docstring describes the global-read footprint **verbatim as a
CuTe layout**:

```python
# tensilelite — SubtileGeometry.py: ABGRGeometry
#   "GR footprint = subtileCount discontiguous strips in M, each
#    (subtileShape[0], subtileShape[1]) MMA tiles, separated by subtileStride
#    MMA tiles. This matches the CuTe layout
#        ((subtileShape[0], subtileCount), subtileShape[1])
#    with stride ((1, subtileStride), ldA)."
#
# for_kernel():  subtileCount  = wg_m
#                subtileStride = (MacroTile // 16) // subtileCount
```

The gaps are the point — the footprint is *not* a permutation. `coalesce`
collapses it only when the strips are actually contiguous.

In [3]:
subtileShape  = (2, 2)   # MMA tiles per strip (M, K)  -- scaled
subtileCount  = 2        # number of strips along M
subtileStride = 4        # M-distance between strips
ldA           = 16       # K (column) stride

gr = Layout(shape=((subtileShape[0], subtileCount), subtileShape[1]),
            stride=((1, subtileStride), ldA))
print("GR footprint:", gr, "| permutation:", is_perm(gr), "(gaps = strided strips)")
gr.pretty_print()
print("coalesced:", coalesce(gr))

GR footprint: ((2,2),2):((1,4),16) | permutation: False (gaps = strided strips)
((2,2),2):((1,4),16)   size=8  cosize=22
        0    1
      ┌────┬────┐
(0,0) │  0 │ 16 │
      ├────┼────┤
(1,0) │  1 │ 17 │
      ╞════╪════╡
(0,1) │  4 │ 20 │
      ├────┼────┤
(1,1) │  5 │ 21 │
      └────┴────┘
coalesced: (2,2,2):(1,4,16)


## 3. Wave-group partition  *(bucket c)*

```python
# tensilelite — SubtileGeometry.py: TileGeometry.localMMATileGrid
def localMMATileGrid(self, macroTile, depthU, waveGroupSize):
    glbl = self.globalMMATileGrid(macroTile, depthU)
    return (glbl[0] // waveGroupSize, glbl[1])    # split the M tiles across waves
```

So a global M-tile index decomposes as `m = localM + waveM · perWaveM`, which is
`column_major(perWaveM, numWavesM)`. Reading it back with `idx2crd` recovers
*(local tile, which wave)*.

In [4]:
perWaveM, numWavesM = 2, 2          # 4 global M tiles, split across 2 waves
part = Layout.column_major(perWaveM, numWavesM)   # (2,2):(1,2)

print("M partition", part, "-- global m -> (localM, waveM):")
for m in range(len(part)):
    print(f"  m={m} -> {part.idx2crd(m)}")
print("\nThe full 2-D wave grid is the product of the M and N partitions",
      "(a nested layout).")

M partition (2,2):(1,2) -- global m -> (localM, waveM):
  m=0 -> (0, 0)
  m=1 -> (1, 0)
  m=2 -> (0, 1)
  m=3 -> (1, 1)

The full 2-D wave grid is the product of the M and N partitions (a nested layout).


## 4. C/D epilogue store tile  *(bucket g)*

```python
# tensilelite — SubtileGeometry.py: CDTileGeometry
def globalMMATileGrid(self, mt0, mt1):
    return (mt0 // instM, mt1 // instN)                    # 16x16 output tiles
def localMMATileGrid(self, mt0, mt1, waveGroup):
    g = self.globalMMATileGrid(mt0, mt1)
    return (g[0] // waveGroup[0], g[1] // waveGroup[1])    # split M AND N
# CD_F32:  storeShape = LoadShape(m=1, k=4)                # 128-bit store = 4 f32 along N
```

A 4-wide f32 store vector tiled over the output grid — `blocked_product` again.

In [5]:
store_vec = Layout.row_major(1, 4)        # 4 contiguous f32 along N (one 128-bit store)
out_grid  = Layout.column_major(2, 2)     # (MT0/16, MT1/16) output tiles

cd = blocked_product(store_vec, out_grid)
print("C/D store layout:", cd, "| permutation:", is_perm(cd))
cd.pretty_print()

C/D store layout: ((1,2),(4,2)):((4,4),(1,8)) | permutation: True
((1,2),(4,2)):((4,4),(1,8))   size=16  cosize=16
        (0,0)   (1,0)   (2,0)   (3,0)   (0,1)   (1,1)   (2,1)   (3,1)
      ┌───────┬───────┬───────┬───────╥───────┬───────┬───────┬───────┐
(0,0) │     0 │     1 │     2 │     3 ║     8 │     9 │    10 │    11 │
      ╞═══════╪═══════╪═══════╪═══════╬═══════╪═══════╪═══════╪═══════╡
(0,1) │     4 │     5 │     6 │     7 ║    12 │    13 │    14 │    15 │
      └───────┴───────┴───────┴───────╨───────┴───────┴───────┴───────┘


## 5. MMA lane → element fragment  *(bucket d)*

```python
# tensilelite — SubtileGeometry.py: MMALayout  /  SubtileLREmit.py: _emitLROffset_TLU0
lane16      = laneId %  instM                 # M row within the MMA tile  (instM = 16)
lane16Group = laneId // instM                 # which K group
kGroups     = waveSize // (instM * blocks)    # = 4  for a 64-lane bf16 fragment
```

That lane → `(row, kGroup)` map is `column_major(16, kGroups)`.

In [6]:
lane = Layout.column_major(16, 4)    # (16,4):(1,16): lane -> (row, kGroup)
print("lane -> (row, kGroup):")
for L in [0, 1, 15, 16, 17, 63]:
    print(f"  lane {L:2d} -> {lane.idx2crd(L)}")
lane.pretty_print()

lane -> (row, kGroup):
  lane  0 -> (0, 0)
  lane  1 -> (1, 0)
  lane 15 -> (15, 0)
  lane 16 -> (0, 1)
  lane 17 -> (1, 1)
  lane 63 -> (15, 3)
(16,4):(1,16)   size=64  cosize=64
      0    1    2    3
   +----+----+----+----+
 0 |  0 | 16 | 32 | 48 |
   +----+----+----+----+
 1 |  1 | 17 | 33 | 49 |
   +----+----+----+----+
 2 |  2 | 18 | 34 | 50 |
   +----+----+----+----+
 3 |  3 | 19 | 35 | 51 |
   +----+----+----+----+
 4 |  4 | 20 | 36 | 52 |
   +----+----+----+----+
 5 |  5 | 21 | 37 | 53 |
   +----+----+----+----+
 6 |  6 | 22 | 38 | 54 |
   +----+----+----+----+
 7 |  7 | 23 | 39 | 55 |
   +----+----+----+----+
 8 |  8 | 24 | 40 | 56 |
   +----+----+----+----+
 9 |  9 | 25 | 41 | 57 |
   +----+----+----+----+
10 | 10 | 26 | 42 | 58 |
   +----+----+----+----+
11 | 11 | 27 | 43 | 59 |
   +----+----+----+----+
12 | 12 | 28 | 44 | 60 |
   +----+----+----+----+
13 | 13 | 29 | 45 | 61 |
   +----+----+----+----+
14 | 14 | 30 | 46 | 62 |
   +----+----+----+----+
15 | 15 | 31 | 47 | 63

## 6. Global-read thread → byte offset  *(bucket b, affine skeleton)*

```python
# tensilelite — SubtileGREmit.py: _emitGROffset_TLU0  (step 5-6)
partitionStride = mmaTileShape[0] * localSubtileGrid[0]
rowOffset = rowId + localRow + partitionRow * partitionStride
GR_offset = colId * loadWidth  +  rowOffset * stride * bpe      # stride = LDA/LDB
# subtile row r adds soffset = (subtileShape[0]*16*bpe) * r * stride
```

With the swizzle on `colId` stripped out (see catalog), that's a 2-mode affine
`Layout` over `(rowId, colId)` — and `enumerate()` reproduces the exact byte
offsets the generated asm uses.

In [7]:
LDA, bpe, loadWidth = 64, 2, 16      # bf16: 128-bit load = 16 bytes = 8 bf16
gr_off = Layout(shape=(4, 4), stride=(LDA * bpe, loadWidth))  # (rowId, colId)

print("GR byte offsets, column-major over (rowId, colId):")
print(gr_off.enumerate())
gr_off.pretty_print()
print("# colId here is the POST-swizzle column; the DPP/permlane swizzle that")
print("# produces it is non-affine -> see the catalog below.")

GR byte offsets, column-major over (rowId, colId):
[0, 128, 256, 384, 16, 144, 272, 400, 32, 160, 288, 416, 48, 176, 304, 432]
(4,4):(128,16)   size=16  cosize=433
      0     1     2     3
  +-----+-----+-----+-----+
0 |   0 |  16 |  32 |  48 |
  +-----+-----+-----+-----+
1 | 128 | 144 | 160 | 176 |
  +-----+-----+-----+-----+
2 | 256 | 272 | 288 | 304 |
  +-----+-----+-----+-----+
3 | 384 | 400 | 416 | 432 |
  +-----+-----+-----+-----+
# colId here is the POST-swizzle column; the DPP/permlane swizzle that
# produces it is non-affine -> see the catalog below.


## 7. Local-read (LDS) byte offset  *(bucket e, affine skeleton)*

```python
# tensilelite — SubtileLREmit.py: _emitLROffset_TLU0
rowOffset = lane16 * subIterKBytes
LR_offset = ((colOffset + advance) % blockSize) * loadWidth   # column
          + rowOffset                                         # lane row
          + waveId * sInterval                                # wave's LDS region
```

The affine skeleton `laneRow · subIterKBytes + dsReadCol · loadWidth + waveId ·
sInterval` is a 3-mode layout (the `% blockSize` wrap and `permlane16` de-swizzle
are the non-affine part — catalog).

In [8]:
subIterKBytes, loadWidthLR, sInterval = 4, 16, 256
lr_off = Layout(shape=(16, 2, 2), stride=(subIterKBytes, loadWidthLR, sInterval))

print("LR offset for (laneRow, dsReadCol, waveId):")
for c in [(0, 0, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1), (15, 1, 1)]:
    print(f"  {c} -> {lr_off[c]} bytes")
print("# The %blockSize column rotation and the permlane16 de-swizzle are")
print("# non-affine -> see the catalog below.")

LR offset for (laneRow, dsReadCol, waveId):
  (0, 0, 0) -> 0 bytes
  (1, 0, 0) -> 4 bytes
  (0, 1, 0) -> 16 bytes
  (0, 0, 1) -> 256 bytes
  (15, 1, 1) -> 332 bytes
# The %blockSize column rotation and the permlane16 de-swizzle are
# non-affine -> see the catalog below.


## 8. wgid → output coordinate  *(bucket h)*

```python
# tensilelite — workgroup schedule (conceptual): decode a linear wg id into an
# output tile coordinate over the macro-tile grid.
tileM, tileN = decode(wg_id, grid)
```

That decode is `idx2crd` on a `blocked_product` schedule (see
`00_layout_concepts.ipynb` §9 for the tiling story in depth).

In [9]:
sched = blocked_product(Layout.row_major(2, 2), Layout.column_major(2, 2))
print("wgid -> coord:")
for w in range(len(sched)):
    print(f"  wgid={w} -> {sched.idx2crd(w)}")

wgid -> coord:
  wgid=0 -> ((0, 0), (0, 0))
  wgid=1 -> ((1, 0), (0, 0))
  wgid=2 -> ((0, 1), (0, 0))
  wgid=3 -> ((1, 1), (0, 0))
  wgid=4 -> ((0, 0), (1, 0))
  wgid=5 -> ((1, 0), (1, 0))
  wgid=6 -> ((0, 1), (1, 0))
  wgid=7 -> ((1, 1), (1, 0))
  wgid=8 -> ((0, 0), (0, 1))
  wgid=9 -> ((1, 0), (0, 1))
  wgid=10 -> ((0, 1), (0, 1))
  wgid=11 -> ((1, 1), (0, 1))
  wgid=12 -> ((0, 0), (1, 1))
  wgid=13 -> ((1, 0), (1, 1))
  wgid=14 -> ((0, 1), (1, 1))
  wgid=15 -> ((1, 1), (1, 1))


## What's left — non-affine options & what they'd need

These subtiling jobs are **not** affine, so a plain shape/stride `Layout` can't
express them yet. Each is shown with its tensilelite snippet and the mosaic
primitive it needs.

### a. LDS bank-conflict swizzles  *(bucket e)* — needs an **XOR / function layout**

```python
# tensilelite — SubtileGREmit.py: _emitGROffset_TLU0  (steps 2-4)
v_mov_b32 colId, colId  dpp:quad_perm:[1,0,3,2]      # swap colId pairs
colId = (colId + waveRotation + intraWaveRotation) % blockSize
# fp8:  colId ^= blockSize // 2
# the LR read inverts this with  v_permlane16_swap_b32
```

These are **XOR / permutation** maps, not `Σ coordᵢ · strideᵢ`. mosaic's C++ has
a `FunctionLayout` notion (gilbert / snake / XOR) but **pymosaic exposes neither
it nor a swizzle layout**. *Need:* an XOR/swizzle layout primitive (or a Python
`FunctionLayout` binding).

### b. Cooperative-load splitting (`loadRatioGR`)  *(bucket c)* — needs **logical_divide**

```python
# tensilelite — Kernel.py: TileInfo.grLoadIndexForSubtile
baseGR = int(linearId // self.loadRatioGR)   # several subtiles share one buffer_load
```

Recovering *which subtiles share a load* is a **quotient** of the subtile grid by
a load-granularity tile — CuTe `logical_divide` (tile + remainder).
`blocked_product` only does the forward tiling. *Need:* `logical_divide` (the C++
version is stubbed and not exposed to Python).

### c. MX scale `op_sel` selection  *(bucket f)* — needs a **bit-interleave**

```python
# tensilelite — Kernel.py: emitMfmaCode
scaleGroup = (mma0 // 2) * subtileKGrid + mmak // subtileKShape
sel        = (mma0 % 2) + 2 * (mmak % 2)     # interleave low bits -> op_sel
```

The grid/subtile decomposition around it *is* affine (the scale **tile** layout is
expressible), but the selector is a bit-interleave, not a stride map.
*Need:* a swizzle / bit-interleave layout.

### d. Fully-fused lane∘subtile∘wave → one offset map  *(buckets b+c+d)* — needs **composition**

The real address is a *composition* of (lane→element) ∘ (subtile placement) ∘
(wave partition) ∘ (swizzle). pymosaic has `blocked_product`, `complement`,
`right_inverse`, `coalesce` — but **general `Layout ∘ Layout` composition is
left to future work** (the C++ `composition` in `algebra.hpp` is partly
stubbed). Two affine layers can be hand-fused by multiplying strides; the swizzle
layer cannot. *Need:* `composition` for `Layout ∘ Layout` (plus the swizzle layout
from (a)).

### Summary

| Job | Bucket | tensilelite source | Missing primitive |
|---|---|---|---|
| LDS bank-conflict swizzle | e | `SubtileGREmit` steps 2–4 | XOR / function layout |
| Cooperative-load split | c | `TileInfo.grLoadIndexForSubtile` | `logical_divide` |
| MX scale `op_sel` | f | `Kernel.emitMfmaCode` | bit-interleave / swizzle |
| Fused lane∘subtile∘wave | b+c+d | whole offset path | `composition` (+ swizzle) |

**Takeaway:** the tile-decomposition + affine-offset half of a tensilelite GEMM
maps onto mosaic layouts *today* (§1–§8). Closing the rest is a short,
well-scoped list: an **XOR/swizzle layout**, **`logical_divide`**, and a
**`composition`** binding.